# Deterministic FlexiTokens vocabularies on FLORES

This is the canonical report for the six-language FLORES analysis. It replaces the earlier seed-based and whitespace-split analysis.

## Method

- Each original FLORES sentence is tokenized as one complete input.
- No seeds or random boundary samples are used.
- A boundary is placed exactly when `sigmoid(boundary_logit) > 0.5`.
- The vocabulary is the counted set of unique decoded output segments across all complete-sentence tokenizations.
- Every vocabulary entry includes up to three source-sentence examples for manual inspection.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, display

RESULTS = Path('results/deterministic_sentence_segment_vocabulary_full_v2')
metadata = json.loads((RESULTS / 'run_metadata.json').read_text())
assert metadata['boundary_rule'] == 'sigmoid(boundary_logit) > 0.5'
assert metadata['randomness'] == 'none; Gumbel sampling is explicitly disabled for this evaluation'
summary = pd.read_csv(RESULTS / 'summary.csv')
assert (summary['sentences'] == 2009).all()
summary[['language', 'sentences', 'unique_output_segments_vocabulary_size', 'total_output_segment_occurrences', 'mean_segments_per_sentence']]

## Summary charts

These figures are generated directly from `summary.csv`.

In [ ]:
display(Image(filename=RESULTS / 'figures' / 'vocabulary_size_by_language.svg'))
display(Image(filename=RESULTS / 'figures' / 'mean_segments_per_sentence.svg'))

## Manual inspection examples

The following are actual stored full-sentence segmentations. Spaces and punctuation shown inside a segment are part of that decoded segment.

In [ ]:
english = pd.read_csv(RESULTS / 'en_sentence_tokenizations.csv')
english.loc[english['text'].str.contains('invention', regex=False), ['text', 'segments', 'segment_count']].head(1)

In [ ]:
for language in ['en', 'es', 'ru', 'uk', 'hi', 'te']:
    vocabulary = pd.read_csv(RESULTS / f'{language}_segment_vocabulary.csv')
    print(f'\n{language}: {len(vocabulary):,} unique output segments')
    display(vocabulary[['segment_display', 'count', 'example_occurrences']].head(5))

## Files

- `summary.csv`: per-language totals.
- `*_segment_vocabulary.csv`: every unique segment, its count, and source examples.
- `*_sentence_tokenizations.csv`: the complete sentence tokenizations used to build the vocabularies.
- `README.md`: method, result table, and interpretation boundaries.